# How to fit spatial tobit panel models

Censored outcomes observed for the same units over several periods.
`SARPanelTobit` puts the spatial lag on the latent outcome; `SEMPanelTobit`
puts the spatial structure in the disturbances.

Both are **NUTS-only**, and both need `N`, `T` and `censoring` stated
explicitly — the panel shape and the censoring threshold are not inferred.
Equations and constructor arguments are in
[Supported Models](../models.md).

In [ ]:
import arviz as az
import numpy as np
import pandas as pd

from neighbayes import dgp
from neighbayes.models import SARPanelTobit, SEMPanelTobit

N, T, SIDE = 100, 5, 10  # 10x10 rook grid, 5 periods
RHO, LAM, SIGMA = 0.35, 0.35, 0.8
BETA = np.array([1.0, 1.4])
FIT = dict(draws=1000, tune=1000, chains=4, random_seed=42, progressbar=False)
rng = np.random.default_rng(42)

## Fit a censored panel with a spatial lag

`y` and `X` are stacked long — $N \cdot T$ rows, unit-major — and you tell the
model the shape with `N` and `T`.

In [ ]:
sar_data = dgp.simulate_panel_sar_tobit_fe(
    N=N,
    T=T,
    n=SIDE,
    contiguity="rook",
    rho=RHO,
    beta=BETA,
    sigma=SIGMA,
    censoring=0.0,
    rng=rng,
)
y_sar, X_sar, W = sar_data["y"], sar_data["X"], sar_data["W_graph"]

print(f"rows          : {len(y_sar)}   ({N} units x {T} periods)")
print(f"censored at 0 : {(y_sar == 0).mean():.1%}")

sar_model = SARPanelTobit(y=y_sar, X=X_sar, W=W, N=N, T=T, censoring=0.0)
sar_idata = sar_model.fit(**FIT)

az.summary(sar_idata, var_names=["rho", "beta", "sigma"]).round(3)

:::{important} `censoring` must match your data
The threshold is not inferred. Pass the value below which observations are
recorded at the limit rather than at their true level — `0.0` here. Get it
wrong, or fit an uncensored model instead, and every coefficient is biased
toward zero because the pile-up at the limit reads as genuine variation.
:::

## Interpreting the coefficients

:::{warning} The effects decomposition is not available for panel tobit
`spatial_effects()` raises `NotImplementedError` on `SARPanelTobit` and
`SEMPanelTobit`. That matters for interpretation, because $\beta$ is **not** a
marginal effect once the outcome is censored — a unit change in $x$ moves the
latent outcome, and only the uncensored part of that reaches the observed one.

Read the coefficients as latent-scale parameters until the decomposition
lands. Where the direct/indirect split is central to your argument, the
cross-sectional [`SARTobit`](tobit_models.ipynb) does provide it.
:::

In [ ]:
try:
    sar_model.spatial_effects()
except NotImplementedError as err:
    print(f"NotImplementedError: {err}")

## Put the spatial structure in the errors instead

`SEMPanelTobit` treats spatial correlation as a nuisance in the disturbances
rather than a channel between units. The spatial parameter is `lam`, and
because there are no covariate-mediated spillovers the effects decomposition
collapses to the coefficient.

In [ ]:
sem_data = dgp.simulate_panel_sem_tobit_fe(
    N=N,
    T=T,
    n=SIDE,
    contiguity="rook",
    lam=LAM,
    beta=BETA,
    sigma=SIGMA,
    censoring=0.0,
    rng=rng,
)
sem_model = SEMPanelTobit(
    y=sem_data["y"],
    X=sem_data["X"],
    W=sem_data["W_graph"],
    N=N,
    T=T,
    censoring=0.0,
)
sem_idata = sem_model.fit(**FIT)

print(f"censored at 0 : {(sem_data['y'] == 0).mean():.1%}")
az.summary(sem_idata, var_names=["lam", "beta", "sigma"]).round(3)

## What to check before trusting the output

The spatial parameter mixes slowest, and these are NUTS models, so divergences
matter. Any at all are worth chasing down before you read the posterior.

In [ ]:
pd.DataFrame(
    [
        pd.Series(
            {
                "spatial parameter": p,
                "posterior mean": float(i.posterior[p].mean()),
                "truth": t,
                "divergences": int(i.sample_stats["diverging"].sum()),
                "min ess_bulk": float(az.summary(i)["ess_bulk"].min()),
                "max rhat": float(az.summary(i)["r_hat"].max()),
            },
            name=n,
        )
        for n, p, t, i in (
            ("SARPanelTobit", "rho", RHO, sar_idata),
            ("SEMPanelTobit", "lam", LAM, sem_idata),
        )
    ]
).round(3)

Short chains are used here so the docs build quickly. If `ess_bulk` on the
spatial parameter is in the low hundreds on your own data, raise `draws` and
`tune` before concluding anything from the interval.

## See also

- [How to fit spatial tobit models](tobit_models.ipynb) — the cross-sectional
  versions, including the Durbin variant
- [Supported Models](../models.md) — equations and constructor arguments
- [How to set priors](priors.ipynb) — `PanelSARTobitPriors`,
  `PanelSEMTobitPriors`